In [4]:
import ROOT
import numpy as np
import pandas as pd
 
 # ROOT.gStyle.SetOptFit(0000)    # aggiunge il label con chi2/ndf e probabilita e parametri

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy()

#definizione variabili
p = df[:, 5]
q = df[:, 6]
err_p = df[:, 7]
err_q = df[:,8]

#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0
Parametri stimati:
A = 1.6119282961553985 ± 5.288587725964002
B = 0.0002970844170377109 ± 49.81764225101172
C = 201.73361492692433 ± 2.7349985423334955
Chi2 = 0.1361327969029265
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =     0.136133
NDf                       =          113
Edm                       =  8.42264e-06
NCalls                    =          502
A                         =      1.61193   +/-   5.28859      	 (limited)
B                         =  0.000297084   +/-   49.8176      	 (limited)
C                         =      201.734   +/-   2.735       


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [5]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if par[0] < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if par[1] < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [6]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff

print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "SR")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")

[-5.42252661e-02 -1.78074296e-02  2.93402927e-03 -9.08321991e-03
 -6.55179272e-03 -5.68381381e-03 -3.94542962e-03 -7.80484695e-03
 -3.57195689e-03 -8.10670497e-03 -1.03050061e-02  1.16387573e-03
 -1.10720037e-02 -7.22336473e-03 -1.19486977e-02  2.58128113e-03
  4.42176289e-03  2.81584133e-03 -7.51477624e-04  1.23378471e-03
 -4.34936104e-03  2.31695936e-03 -2.11406822e-04  5.70839955e-03
 -1.26433867e-04 -2.19551737e-03 -7.13192700e-03 -8.66484276e-03
 -8.22337736e-04  1.39288486e-03 -3.70111847e-03  6.83317492e-03
  3.90366688e-04 -3.19490680e-04 -4.45015498e-04  1.36814778e-03
  1.24890639e-03  5.36766055e-03  3.81496916e-03  1.22578585e-02
  1.52536120e-03  6.39177723e-03  4.52414555e-03 -1.14388383e-02
 -1.50014809e-02  1.05406809e-03 -7.92985717e-03 -4.12206583e-03
  5.27290226e-03 -2.71026947e-03 -9.23950574e-04  7.88446507e-03
 -1.37921913e-02  3.73030856e-03 -1.25754011e-03  9.86406060e-03
  4.52989150e-03  8.36689374e-03 -6.26063256e-03 -1.44712714e-03
  3.81139219e-03 -1.01159

ReferenceError: attempt to access a null-pointer

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <ROOT::Fit::FillData>: fit range is outside histogram range, no fit data for xaxis
Warning in <Fit>: Fit data is empty 
Error in <TFitResultPtr>: TFitResult is empty - use the fit option S


In [ ]:
if parameters[0] - 1 < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if parameters[1] < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if parameters[2] - 1 < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const è compatibile con 1
---
mu è compatibile con 0
---
sigma è compatibile con 1


In [ ]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

Fit status: 0
Parametri stimati:
m = -0.9582207595159988 ± 0.0006059688593299256
q = 634.227402598752 ± 0.23136647614925315
Chi2 = 243717057.91991618
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  2.43717e+08
NDf                       =          122
Edm                       =  2.46915e-15
NCalls                    =           31
m                         =    -0.958221   +/-   0.000605969 
q                         =      634.227   +/-   0.231366    


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
